In [ ]:
import json
import sys
from rich import print as rprint
from pathlib import Path
import re
import subprocess
from datetime import datetime
from collections import defaultdict

nb_dir = Path.cwd()
project_root = nb_dir.parent.parent
sys.path.insert(0, str(project_root))

In [ ]:
db_books_file = Path(project_root / "data_reload/db_files/books.json")
db_b2p_file = Path(project_root / "data_reload/db_files/books2people.json")
db_admin_file = Path(project_root / "data_reload/db_files/book_admin.json")
db_people_file = Path(project_root / "data_reload/db_files/people.json")

with open(db_books_file, "r") as f:
   books = json.load(f)
with open(db_b2p_file, "r") as f:
   b2p = json.load(f)
with open(db_admin_file, "r") as f:
   admin = json.load(f)
with open(db_people_file, "r") as f:
   people = json.load(f)


books_dict = {i["composite_id"]: i for i in books}
b2p_dict = {i["composite_id"]: i for i in b2p}
people_dict = {i["unified_id"]: i for i in people}

with open("remaining_cids.json") as f:
    remaining_cids = set(json.load(f))

search_path = Path(project_root / "data/people/")


In [ ]:
# found_in = {}

# for i, cid in enumerate(remaining_cids):
#     if i == 10:
#         break
#     result = subprocess.run(
#         ["grep", "-rl", cid, search_path], capture_output=True, text=True
#     )
#     if result.stdout.strip():
#         result_lines = result.stdout.strip().splitlines()
#         result_path = [str(Path(f).relative_to(project_root)) for f in result_lines]
#         found_in[cid] = result_path
#         print(result_path)

# print(f"{len(found_in)} of {len(remaining_cids)} found")


# V1 VERY SLOW, no outputs, I'm too impatient

# for cid in remaining_cids:
#     result = subprocess.run(
#         ["grep", "-rl", cid, search_path], capture_output=True, text=True
#     )
#     if result.stdout.strip():
#         found_in[cid] = result.stdout.strip().splitlines()
# print(f"{len(found_in)} of {len(remaining_cids)} found")


In [ ]:
# claude version

found_cids = set()
not_found_cids = set()
found_in = {}
cids_per_file = {}

for i, cid in enumerate(remaining_cids):
    result = subprocess.run(
        ["grep", "-rl", cid, search_path], capture_output=True, text=True
    )
    if result.stdout.strip():
        result_lines = result.stdout.strip().splitlines()
        result_paths = [str(Path(f).relative_to(project_root)) for f in result_lines]
        found_in[cid] = result_paths
        found_cids.add(cid)
        for filepath in result_paths:
            cids_per_file[filepath] = cids_per_file.get(filepath, 0) + 1
    else:
        not_found_cids.add(cid)

    if (i + 1) % 10 == 0:
        print(f"{i + 1} checked | found: {len(found_cids)} | not found: {len(not_found_cids)}")


In [ ]:
print(f"\nDone. {len(found_cids)} found, {len(not_found_cids)} not found")
for filepath, count in sorted(cids_per_file.items(), key=lambda x: x[1], reverse=True):
    print(f"  {filepath}: {count}")

with open("found_cids.json", "w") as f:
    json.dump(list(found_cids), f, ensure_ascii=False, indent=2)
with open("not_found_cids.json", "w") as f:
    json.dump(list(not_found_cids), f, ensure_ascii=False, indent=2)
with open("found_in.json", "w") as f:
    json.dump(found_in, f, ensure_ascii=False, indent=2)
with open("cids_per_file.json", "w") as f:
    json.dump(cids_per_file, f, ensure_ascii=False, indent=2)
